# Validation: Silver source reconciliation

## Purpose

Validate flood-control component preservation and exact Contract ID reconciliation before either output is used downstream. Results cover consistency, accuracy, completeness, auditability, validity, uniqueness, and timeliness. Evidence is persisted before the blocking gate runs.


## 1. Build reusable metrics

The metric view scans each bounded Silver output once and keeps invalid IDs, repeated components, unmatched groups, ambiguous targets, cost conflicts, and lineage exceptions visible.


In [0]:
%sql
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW silver_source_reconciliation_metrics AS
SELECT
    (SELECT COUNT(*) FROM `01-bronze`.flood_control_projects) AS bronze_rows,
    (SELECT COUNT(*) FROM `02-silver`.silver_flood_control_component) AS component_rows,
    (SELECT COUNT(DISTINCT flood_component_key)
     FROM `02-silver`.silver_flood_control_component) AS distinct_component_keys,
    (SELECT COUNT_IF(source_feature_id IS NULL OR TRIM(source_feature_id) = '')
     FROM `02-silver`.silver_flood_control_component) AS missing_source_feature_ids,
    (SELECT COUNT_IF(source_contract_id_normalized IS NULL)
     FROM `02-silver`.silver_flood_control_component) AS invalid_source_contract_ids,
    (SELECT COUNT_IF(source_version IS NULL OR TRIM(source_version) = '')
     FROM `02-silver`.silver_flood_control_component) AS missing_source_version_rows,
    (SELECT COUNT(*)
     FROM (
         SELECT source_contract_id_normalized
         FROM `02-silver`.silver_flood_control_component
         WHERE source_contract_id_normalized IS NOT NULL
         GROUP BY source_contract_id_normalized
         HAVING COUNT(*) > 1
     )) AS repeated_source_contract_groups,
    (SELECT COUNT_IF(source_contract_cost_status = 'UNPARSEABLE')
     FROM `02-silver`.silver_flood_control_component) AS cost_parse_failures,
    (SELECT COUNT_IF(source_contract_cost_status = 'NEGATIVE')
     FROM `02-silver`.silver_flood_control_component) AS negative_cost_rows,
    (SELECT COUNT_IF(coordinate_status <> 'VALID_PAIR')
     FROM `02-silver`.silver_flood_control_component) AS coordinate_exception_rows,
    (SELECT COUNT_IF(
        source_snapshot_id IS NULL OR TRIM(source_snapshot_id) = ''
        OR source_ingest_run_id IS NULL OR TRIM(source_ingest_run_id) = ''
        OR source_load_ts IS NULL OR source_system IS NULL OR TRIM(source_system) = ''
     ) FROM `02-silver`.silver_flood_control_component) AS missing_component_lineage,
    (SELECT COUNT(DISTINCT source_snapshot_id)
     FROM `02-silver`.silver_flood_control_component) AS component_snapshot_count,
    (SELECT COUNT(DISTINCT run_id)
     FROM `02-silver`.silver_flood_control_component) AS component_run_count,
    (SELECT COUNT(*) FROM `02-silver`.silver_project_source_match) AS match_rows,
    (SELECT COUNT(DISTINCT source_contract_id_normalized)
     FROM `02-silver`.silver_flood_control_component
     WHERE source_contract_id_normalized IS NOT NULL) AS usable_source_contract_groups,
    (SELECT COUNT(DISTINCT project_source_match_key)
     FROM `02-silver`.silver_project_source_match) AS distinct_match_keys,
    (SELECT COALESCE(SUM(source_component_count), 0)
     FROM `02-silver`.silver_project_source_match) AS matched_table_component_accounting,
    (SELECT COUNT_IF(match_status = 'MATCHED_EXACT_CONTRACT_ID')
     FROM `02-silver`.silver_project_source_match) AS matched_groups,
    (SELECT COUNT_IF(match_status = 'UNMATCHED')
     FROM `02-silver`.silver_project_source_match) AS unmatched_groups,
    (SELECT COUNT_IF(match_status = 'AMBIGUOUS_TARGET')
     FROM `02-silver`.silver_project_source_match) AS ambiguous_groups,
    (SELECT COUNT_IF(source_cost_resolution_status = 'CONFLICT')
     FROM `02-silver`.silver_project_source_match) AS source_cost_conflict_groups,
    (SELECT COUNT_IF(
        (match_status = 'MATCHED_EXACT_CONTRACT_ID'
         AND (target_candidate_count <> 1
              OR matched_project_key IS NULL
              OR matched_project_contract_id IS NULL
              OR UPPER(TRIM(matched_project_contract_id))
                 <> source_contract_id_normalized))
        OR (match_status <> 'MATCHED_EXACT_CONTRACT_ID'
            AND (matched_project_key IS NOT NULL
                 OR matched_project_contract_id IS NOT NULL))
     ) FROM `02-silver`.silver_project_source_match) AS invalid_match_assignments,
    (SELECT COUNT_IF(
        source_cost_resolution_status = 'CONFLICT'
        AND resolved_source_contract_cost_pesos IS NOT NULL
     ) FROM `02-silver`.silver_project_source_match) AS invalid_cost_resolutions,
    (SELECT COUNT_IF(
        source_snapshot_id IS NULL OR TRIM(source_snapshot_id) = ''
        OR source_component_run_id IS NULL OR TRIM(source_component_run_id) = ''
        OR target_project_run_id IS NULL OR TRIM(target_project_run_id) = ''
        OR run_id IS NULL OR TRIM(run_id) = ''
     ) FROM `02-silver`.silver_project_source_match) AS missing_match_lineage,
    (SELECT COUNT(DISTINCT run_id)
     FROM `02-silver`.silver_project_source_match) AS match_run_count;


## 2. Create and persist checks

Blocking checks protect grain, row accounting, exact-match correctness, cost-resolution safety, and lineage. Expected source imperfections remain nonblocking flags.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_source_reconciliation_context AS
WITH component_context AS (
    SELECT
        MAX(source_snapshot_id) AS source_snapshot_id,
        MAX(run_id) AS component_run_id
    FROM `02-silver`.silver_flood_control_component
),
project_context AS (
    SELECT MAX(run_id) AS project_run_id
    FROM `02-silver`.silver_project
),
match_context AS (
    SELECT MAX(run_id) AS match_run_id
    FROM `02-silver`.silver_project_source_match
)
SELECT
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(component.source_snapshot_id, ''),
            COALESCE(component.component_run_id, ''),
            COALESCE(project.project_run_id, ''),
            COALESCE(match.match_run_id, ''),
            'silver_source_reconciliation_validation_v2'
        ),
        256
    ) AS validation_run_id,
    COALESCE(
        match.match_run_id,
        SHA2(
            CONCAT_WS(
                '||',
                COALESCE(component.component_run_id, ''),
                COALESCE(project.project_run_id, ''),
                'silver_project_source_match_v1'
            ),
            256
        )
    ) AS pipeline_run_id,
    component.source_snapshot_id,
    'silver_source_reconciliation_validation_v2' AS rule_version,
    CURRENT_TIMESTAMP() AS run_ts
FROM component_context AS component
CROSS JOIN project_context AS project
CROSS JOIN match_context AS match;

CREATE OR REPLACE TEMPORARY VIEW silver_source_reconciliation_checks AS
WITH raw_checks AS (
    SELECT 'silver_flood_control_component' AS table_name, 'Completeness' AS data_quality_attribute,
        'Bronze flood rows reconcile to Silver component rows' AS data_quality_check,
        ABS(component_rows - bronze_rows) AS failed_rows, bronze_rows AS total_rows,
        component_rows = bronze_rows AS passed, 'stop' AS action,
        CONCAT('Bronze rows=', bronze_rows, '; component rows=', component_rows) AS details
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Uniqueness',
        'Component keys are unique at source object grain',
        component_rows - distinct_component_keys, component_rows,
        component_rows = distinct_component_keys, 'stop',
        CONCAT('Distinct keys=', distinct_component_keys)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Validity',
        'Every component retains a source feature identifier',
        missing_source_feature_ids, component_rows, missing_source_feature_ids = 0, 'stop',
        'Source feature identity comes from Bronze object_id'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Timeliness',
        'Component output contains one selected snapshot and run',
        ABS(component_snapshot_count - 1) + ABS(component_run_count - 1), 2,
        component_snapshot_count = 1 AND component_run_count = 1, 'stop',
        CONCAT('Snapshots=', component_snapshot_count, '; runs=', component_run_count)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Auditability',
        'Mandatory component lineage is present', missing_component_lineage, component_rows,
        missing_component_lineage = 0, 'stop', 'Checks source, snapshot, ingest, and load timestamp'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Completeness',
        'Invalid source Contract IDs remain explicitly accounted for',
        invalid_source_contract_ids, component_rows, invalid_source_contract_ids = 0, 'flag',
        'Rows stay in the component table and are excluded only from the match table grain'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Auditability',
        'Optional source version gaps remain visible', missing_source_version_rows,
        component_rows, missing_source_version_rows = 0, 'flag',
        'Snapshot and ingest lineage remain mandatory even when version is unavailable'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Consistency',
        'Repeated source Contract IDs remain visible', repeated_source_contract_groups,
        usable_source_contract_groups, repeated_source_contract_groups = 0, 'flag',
        'Repeated Contract IDs are preserved; this is not a deduplication rule'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Accuracy',
        'Source contract cost parses safely', cost_parse_failures, component_rows,
        cost_parse_failures = 0, 'flag', 'Raw cost is retained beside TRY_CAST output'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Validity',
        'Source contract cost is nonnegative when parsed', negative_cost_rows, component_rows,
        negative_cost_rows = 0, 'flag', 'Negative parsed costs require review'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_flood_control_component', 'Completeness',
        'Coordinate exceptions remain visible', coordinate_exception_rows, component_rows,
        coordinate_exception_rows = 0, 'flag', 'No geography is assigned in this milestone'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Completeness',
        'Every usable source Contract ID has one match result',
        ABS(match_rows - usable_source_contract_groups), usable_source_contract_groups,
        match_rows = usable_source_contract_groups, 'stop',
        CONCAT('Match rows=', match_rows, '; usable groups=', usable_source_contract_groups)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Uniqueness',
        'Match keys are unique at normalized Contract ID and run grain',
        match_rows - distinct_match_keys, match_rows, match_rows = distinct_match_keys, 'stop',
        CONCAT('Distinct match keys=', distinct_match_keys)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Consistency',
        'Component accounting includes matched groups and invalid IDs',
        ABS(component_rows - matched_table_component_accounting - invalid_source_contract_ids),
        component_rows,
        component_rows = matched_table_component_accounting + invalid_source_contract_ids, 'stop',
        CONCAT('Grouped components=', matched_table_component_accounting,
               '; invalid IDs=', invalid_source_contract_ids)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Accuracy',
        'Exact matches and unresolved results never receive arbitrary targets',
        invalid_match_assignments, match_rows, invalid_match_assignments = 0, 'stop',
        'Matched Contract IDs must equal after TRIM and UPPER only'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Validity',
        'Conflicting source costs never publish a resolved cost',
        invalid_cost_resolutions, match_rows, invalid_cost_resolutions = 0, 'stop',
        'Repeated cost values are never blindly summed'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Auditability',
        'Match lineage is complete', missing_match_lineage, match_rows,
        missing_match_lineage = 0, 'stop', 'Checks source, component, target, and match runs'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Timeliness',
        'Match output contains one deterministic run', ABS(match_run_count - 1), 1,
        match_run_count = 1, 'stop', CONCAT('Match runs=', match_run_count)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Completeness',
        'Unmatched source Contract IDs remain visible', unmatched_groups, match_rows,
        unmatched_groups = 0, 'flag', 'Unmatched records require review; they are not dropped'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Uniqueness',
        'Ambiguous target Contract IDs remain visible', ambiguous_groups, match_rows,
        ambiguous_groups = 0, 'flag', 'No target is selected when candidate count exceeds one'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Consistency',
        'Conflicting source cost groups remain visible', source_cost_conflict_groups, match_rows,
        source_cost_conflict_groups = 0, 'flag', 'No project budget is created from source costs'
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT 'silver_project_source_match', 'Accuracy',
        'Exact Contract ID matches are reported separately', 0, match_rows, TRUE, 'flag',
        CONCAT('Matched=', matched_groups, '; unmatched=', unmatched_groups,
               '; ambiguous=', ambiguous_groups)
    FROM silver_source_reconciliation_metrics
    UNION ALL
    SELECT
        'silver_project_source_match',
        'Consistency',
        'Match statuses reconcile to all usable source Contract IDs',
        ABS(
            usable_source_contract_groups
            - matched_groups
            - unmatched_groups
            - ambiguous_groups
        ),
        usable_source_contract_groups,
        matched_groups + unmatched_groups + ambiguous_groups
            = usable_source_contract_groups,
        'stop',
        CONCAT(
            'Matched=', matched_groups,
            '; unmatched=', unmatched_groups,
            '; ambiguous=', ambiguous_groups,
            '; usable groups=', usable_source_contract_groups
        )
    FROM silver_source_reconciliation_metrics
),
final_checks AS (
    SELECT
        context.validation_run_id, context.pipeline_run_id, 'silver' AS layer,
        raw.table_name, context.source_snapshot_id, context.rule_version,
        raw.data_quality_attribute, raw.data_quality_check,
        CAST(raw.failed_rows AS BIGINT) AS failed_rows,
        CAST(raw.total_rows AS BIGINT) AS total_rows,
        CASE WHEN raw.total_rows = 0 THEN NULL
             ELSE 100.0 * raw.failed_rows / raw.total_rows END AS percentage,
        CASE WHEN raw.passed THEN 'PASS'
             WHEN raw.action = 'stop' THEN 'FAIL' ELSE 'FLAG' END AS status,
        raw.action, raw.details, context.run_ts
    FROM raw_checks AS raw
    CROSS JOIN silver_source_reconciliation_context AS context
)
SELECT * FROM final_checks;

CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL,
    pipeline_run_id STRING NOT NULL,
    layer STRING NOT NULL,
    table_name STRING NOT NULL,
    source_snapshot_id STRING,
    rule_version STRING,
    data_quality_attribute STRING NOT NULL,
    data_quality_check STRING NOT NULL,
    failed_rows BIGINT,
    total_rows BIGINT,
    percentage DOUBLE,
    status STRING NOT NULL,
    action STRING NOT NULL,
    details STRING,
    run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING silver_source_reconciliation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Silver source reconciliation checks failed'
    )
)
FROM silver_source_reconciliation_checks;


## 3. Review match coverage and retained evidence

Coverage uses usable normalized source Contract ID groups as the denominator. Component counts remain separate so repeated source records are never mistaken for distinct projects.


In [0]:
%sql
SELECT
    match_status,
    COUNT(*) AS source_contract_groups,
    SUM(source_component_count) AS source_components,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS group_percentage
FROM `02-silver`.silver_project_source_match
GROUP BY match_status
ORDER BY source_contract_groups DESC;

SELECT
    source_cost_resolution_status,
    COUNT(*) AS source_contract_groups,
    SUM(source_component_count) AS source_components
FROM `02-silver`.silver_project_source_match
GROUP BY source_cost_resolution_status
ORDER BY source_contract_groups DESC;

SELECT
    validation_run_id, table_name, data_quality_attribute, status, action,
    failed_rows, total_rows, percentage, data_quality_check, details
FROM silver_source_reconciliation_checks
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END,
    table_name, data_quality_check;


# Summary

The validator persists evidence to `04-validation`.silver_dq_results before enforcing the stop gate. It reports exact-match coverage, invalid IDs, repeated components, unresolved targets, conflicting costs, and lineage without changing either Silver table.

Requires Databricks execution.
